In [2]:
import numpy as np
import pandas as pd
import time
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import time
import os 
import sys
sys.path.append('..')


In [3]:
data_train = pd.read_parquet('../temp/X_train_v1.parquet')
data_val = pd.read_parquet('../temp/X_val_v1.parquet')



In [4]:
data_train.shape

(452266, 895)

In [5]:
data_val.shape

(61502, 895)

In [6]:
X_train = data_train.iloc[1:100001]
X_val = data_val.iloc[1:10001]

In [7]:
y_train = X_train['TARGET']
y_val = X_val['TARGET']
X_train = X_train.drop(columns=['TARGET'])
X_val = X_val.drop(columns=['TARGET'])

In [8]:
feature_selection_plan = [
    {'start': 895, 'end': 700, 'steps': 3},
    {'start': 700, 'end': 500, 'steps': 2}
]

current_features = X_train.columns if isinstance(X_train, pd.DataFrame) else np.arange(X_train.shape[1])
feature_importance_df = pd.DataFrame(index=current_features)

for phase in feature_selection_plan:
    start = phase['start']
    end = phase['end']
    steps = phase['steps']
    
    features_to_remove = start - end
    step_size = features_to_remove // steps 
    
    print(f"\nStarting feature reduction from {start} to {end} features in {steps} step(s) with step size {step_size}.")

    for step in range(1, steps + 1):
        n_features_to_select = start - step_size * step
        n_features_to_select = max(n_features_to_select, end) 

        print(f" RFE Step {step}: Selecting {n_features_to_select} features.")
        estimator = LogisticRegression(
            penalty='l2',
            solver='saga',
            max_iter=1000,
            n_jobs=-1, 
            warm_start=True
        )
        selector = RFE(
            estimator=estimator,
            n_features_to_select=n_features_to_select,
            step=step_size,
            verbose=1
        )
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('rfe', selector)
        ])
        print("    Fitting RFE...")
        start_time = time.time()
        pipeline.fit(X_train[current_features], y_train)
        end_time_step = time.time()
        print(f"    RFE completed in {end_time_step - start_time:.2f} seconds.")

        selected_features_mask = pipeline.named_steps['rfe'].support_
        selected_features = current_features[selected_features_mask] if isinstance(current_features, pd.Index) else current_features[selected_features_mask]
        print(f"    Number of selected features: {selected_features_mask.sum()}")

        coefficients = pipeline.named_steps['rfe'].estimator_.coef_.flatten()
        feature_importance_df = feature_importance_df.loc[selected_features]
        feature_importance_df['importance'] = np.abs(coefficients)

        current_features = selected_features

print("\nFinal number of selected features:", len(current_features))
print("Final selected features:", current_features.tolist())

if isinstance(X_train, pd.DataFrame):
    X_train_selected = X_train[current_features]
    X_test_selected = X_val[current_features]
else:
    feature_indices = current_features
    X_train_selected = X_train[:, feature_indices]
    X_test_selected = X_val[:, feature_indices]

scaler = StandardScaler()
X_train_selected = scaler.fit_transform(X_train_selected)
X_test_selected = scaler.transform(X_test_selected)

final_model = LogisticRegression(
    penalty='l2',
    solver='saga',
    max_iter=1000,
    n_jobs=-1
)

print("\nTraining the final model with selected features...")
final_model.fit(X_train_selected, y_train)

# Evaluate the model
y_pred = final_model.predict(X_test_selected)
accuracy = accuracy_score(y_val, y_pred)
print(f"Final model accuracy with selected features: {accuracy:.4f}")

# Save the selected features and their importance
if isinstance(X_train, pd.DataFrame):
    feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)
    print("\nSelected Features and their Importance:")
    print(feature_importance_df)
    feature_importance_df.to_csv('500feat.csv', index=True)



Starting feature reduction from 895 to 700 features in 3 step(s) with step size 65.
 RFE Step 1: Selecting 830 features.
    Fitting RFE...
Fitting estimator with 894 features.


/home/quanghung20gg/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/quanghung20gg/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


    RFE completed in 748.31 seconds.
    Number of selected features: 830
 RFE Step 2: Selecting 765 features.
    Fitting RFE...
Fitting estimator with 830 features.


/home/quanghung20gg/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/quanghung20gg/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


    RFE completed in 673.92 seconds.
    Number of selected features: 765
 RFE Step 3: Selecting 700 features.
    Fitting RFE...
Fitting estimator with 765 features.


...

/home/quanghung20gg/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/quanghung20gg/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


    RFE completed in 562.77 seconds.
    Number of selected features: 600
 RFE Step 2: Selecting 500 features.
    Fitting RFE...
Fitting estimator with 600 features.


/home/quanghung20gg/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/quanghung20gg/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


    RFE completed in 495.96 seconds.
    Number of selected features: 500

Final number of selected features: 500
Final selected features: ['NAME_INCOME_TYPE_Commercial associate', 'NAME_INCOME_TYPE_State servant', 'NAME_INCOME_TYPE_Working', 'NAME_FAMILY_STATUS_Civil marriage', 'NAME_FAMILY_STATUS_Widow', 'ORGANIZATION_TYPE_Business Entity', 'ORGANIZATION_TYPE_Construction', 'ORGANIZATION_TYPE_Realtor', 'ORGANIZATION_TYPE_Self-employed', 'AMT_INCOME_TOTAL', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE', 'COMMONAREA_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'APARTMENTS_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI', 'COMMONAREA_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI', 'TOTALAREA_MODE', 'BUREAU_GENERAL_AMT_CREDIT_SUM_sum', 'BUREAU_GENERAL_AMT_CREDIT_SUM_mean', 'BUREAU_GENERAL_AMT_CREDIT_SU

/home/quanghung20gg/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [9]:
feature_selection_plan = [
    {'start': 895, 'end': 700, 'steps': 3},
    {'start': 700, 'end': 500, 'steps': 2},
    {'start': 500, 'end': 400, 'steps': 1}
]

current_features = X_train.columns if isinstance(X_train, pd.DataFrame) else np.arange(X_train.shape[1])
feature_importance_df = pd.DataFrame(index=current_features)

for phase in feature_selection_plan:
    start = phase['start']
    end = phase['end']
    steps = phase['steps']
    
    features_to_remove = start - end
    step_size = features_to_remove // steps 
    
    print(f"\nStarting feature reduction from {start} to {end} features in {steps} step(s) with step size {step_size}.")

    for step in range(1, steps + 1):
        n_features_to_select = start - step_size * step
        n_features_to_select = max(n_features_to_select, end) 

        print(f" RFE Step {step}: Selecting {n_features_to_select} features.")
        estimator = LogisticRegression(
            penalty='l2',
            solver='saga',
            max_iter=1000,
            n_jobs=-1, 
            warm_start=True
        )
        selector = RFE(
            estimator=estimator,
            n_features_to_select=n_features_to_select,
            step=step_size,
            verbose=1
        )
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('rfe', selector)
        ])
        print("    Fitting RFE...")
        start_time = time.time()
        pipeline.fit(X_train[current_features], y_train)
        end_time_step = time.time()
        print(f"    RFE completed in {end_time_step - start_time:.2f} seconds.")

        selected_features_mask = pipeline.named_steps['rfe'].support_
        selected_features = current_features[selected_features_mask] if isinstance(current_features, pd.Index) else current_features[selected_features_mask]
        print(f"    Number of selected features: {selected_features_mask.sum()}")

        coefficients = pipeline.named_steps['rfe'].estimator_.coef_.flatten()
        feature_importance_df = feature_importance_df.loc[selected_features]
        feature_importance_df['importance'] = np.abs(coefficients)

        current_features = selected_features

print("\nFinal number of selected features:", len(current_features))
print("Final selected features:", current_features.tolist())

if isinstance(X_train, pd.DataFrame):
    X_train_selected = X_train[current_features]
    X_test_selected = X_val[current_features]
else:
    feature_indices = current_features
    X_train_selected = X_train[:, feature_indices]
    X_test_selected = X_val[:, feature_indices]

scaler = StandardScaler()
X_train_selected = scaler.fit_transform(X_train_selected)
X_test_selected = scaler.transform(X_test_selected)

final_model = LogisticRegression(
    penalty='l2',
    solver='saga',
    max_iter=1000,
    n_jobs=-1
)

print("\nTraining the final model with selected features...")
final_model.fit(X_train_selected, y_train)

# Evaluate the model
y_pred = final_model.predict(X_test_selected)
accuracy = accuracy_score(y_val, y_pred)
print(f"Final model accuracy with selected features: {accuracy:.4f}")

# Save the selected features and their importance
if isinstance(X_train, pd.DataFrame):
    feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)
    print("\nSelected Features and their Importance:")
    print(feature_importance_df)
    feature_importance_df.to_csv('400feat.csv', index=True)



Starting feature reduction from 895 to 700 features in 3 step(s) with step size 65.
 RFE Step 1: Selecting 830 features.
    Fitting RFE...


Fitting estimator with 894 features.


/home/quanghung20gg/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/quanghung20gg/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


    RFE completed in 729.06 seconds.
    Number of selected features: 830
 RFE Step 2: Selecting 765 features.
    Fitting RFE...
Fitting estimator with 830 features.


/home/quanghung20gg/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/quanghung20gg/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


...

/home/quanghung20gg/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/quanghung20gg/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


    RFE completed in 567.27 seconds.
    Number of selected features: 500

Starting feature reduction from 500 to 400 features in 1 step(s) with step size 100.
 RFE Step 1: Selecting 400 features.
    Fitting RFE...
Fitting estimator with 500 features.


/home/quanghung20gg/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/quanghung20gg/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


    RFE completed in 428.91 seconds.
    Number of selected features: 400

Final number of selected features: 400
Final selected features: ['NAME_INCOME_TYPE_Commercial associate', 'NAME_INCOME_TYPE_Working', 'NAME_FAMILY_STATUS_Civil marriage', 'ORGANIZATION_TYPE_Business Entity', 'ORGANIZATION_TYPE_Construction', 'ORGANIZATION_TYPE_Self-employed', 'AMT_INCOME_TOTAL', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'BASEMENTAREA_MODE', 'COMMONAREA_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 'NONLIVINGAPARTMENTS_MODE', 'APARTMENTS_MEDI', 'YEARS_BEGINEXPLUATATION_MEDI', 'COMMONAREA_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'TOTALAREA_MODE', 'BUREAU_GENERAL_AMT_CREDIT_SUM_sum', 'BUREAU_GENERAL_AMT_CREDIT_SUM_mean', 'BUREAU_GENERAL_AMT_CREDIT_SUM_DEBT_sum', 'BUREAU_GENERAL_AMT_CREDIT_SUM_DEBT_mean', 'BUREAU_GENERAL_AMT_CREDIT_SUM_OVERDUE_sum', 'BUREAU_GENERAL_AMT_CREDIT_MAX_OVERDUE_sum', 'BUREAU_GENERAL_AMT_C

/home/quanghung20gg/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
